# Day 2 — Iterators, Generators & Memory-Efficient Data Handling

## Objective
Demonstrate memory-efficient data loading primitives. Compare eager evaluation ($O(N)$ memory) with custom lazy batch iteration ($O(\text{Batch Size})$ memory) using Python's iteration protocol (`__iter__`, `__next__`, `yield`) and measure peak memory consumption with `tracemalloc` across datasets up to 10,000 rows.

## 1. Iteration Protocol Concepts

- **`__iter__()`**: Returns the iterator object itself.
- **`__next__()`**: Returns the next batch or raises `StopIteration` when data stream is exhausted.
- **`yield` Generators**: Pauses function execution and yields data lazily without keeping the entire dataset in RAM.
- **Out-of-Core Computing**: Enables processing massive machine learning datasets exceeding RAM limits.

In [1]:
import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from task_analytics.data_iterator import CSVBatchIterator, load_csv_eager
print('CSVBatchIterator imported successfully.')

CSVBatchIterator imported successfully.


## 2. Sample Dataset Generation

Generate synthetic CSV task datasets (100, 1,000, and 10,000 rows) under `data/raw/`.

In [2]:
sys.path.insert(0, str(repo_root / 'scripts'))
from generate_sample_data import generate_csv_data

raw_dir = repo_root / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

sample_files = {}
for size in [100, 1000, 10000]:
    p = raw_dir / f'sample_{size}.csv'
    generate_csv_data(p, size)
    sample_files[size] = p

Successfully generated 100 rows -> C:\Users\bc\task-management-api\data\raw\sample_100.csv
Successfully generated 1,000 rows -> C:\Users\bc\task-management-api\data\raw\sample_1000.csv
Successfully generated 10,000 rows -> C:\Users\bc\task-management-api\data\raw\sample_10000.csv


## 3. Eager vs Lazy Loading Demonstration

Compare loading an entire CSV into memory list vs streaming with `CSVBatchIterator`.

In [3]:
csv_100 = sample_files[100]
# Eager loading
eager_data = load_csv_eager(csv_100)
print(f'Eagerly loaded {len(eager_data)} rows into a Python list.')
print(f'Sample Row 1: {eager_data[0]}')

# Lazy batch iteration
iterator = CSVBatchIterator(csv_100, batch_size=25)
print('\nLazy Iteration Batches:')
for idx, batch in enumerate(iterator, 1):
    print(f'  Batch {idx}: {len(batch)} items yielded lazily')

Eagerly loaded 100 rows into a Python list.
Sample Row 1: {'task_id': 'TASK-000001', 'title': 'Feature implementation for module 1', 'priority': 'medium', 'status': 'todo', 'estimated_hours': '2.0', 'logged_hours': '0.8', 'created_at': '2026-01-01T09:05:00'}

Lazy Iteration Batches:
  Batch 1: 25 items yielded lazily
  Batch 2: 25 items yielded lazily
  Batch 3: 25 items yielded lazily
  Batch 4: 25 items yielded lazily


## 4. Memory Profiling Benchmark (`tracemalloc`)

Measure peak memory consumption for Eager vs Lazy loading across 100, 1,000, and 10,000 row datasets.

In [4]:
import gc
import tracemalloc

def profile_eager(filepath):
    gc.collect()
    tracemalloc.reset_peak()
    tracemalloc.start()
    data = load_csv_eager(filepath)
    count = sum(1 for row in data)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024  # KB

def profile_lazy(filepath, batch_size=100):
    gc.collect()
    tracemalloc.reset_peak()
    tracemalloc.start()
    iterator = CSVBatchIterator(filepath, batch_size=batch_size)
    count = 0
    for batch in iterator:
        count += len(batch)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024  # KB

print(f'{"Dataset Rows":<15} | {"Eager Peak (KB)":<18} | {"Lazy Peak (KB)":<18} | {"Memory Savings":<15}')
print('-' * 72)
for size in [100, 1000, 10000]:
    p = sample_files[size]
    eager_kb = profile_eager(p)
    lazy_kb = profile_lazy(p, batch_size=100)
    savings = (1 - (lazy_kb / eager_kb)) * 100
    print(f'{size:<15} | {eager_kb:18.2f} | {lazy_kb:18.2f} | {savings:14.1f}%')

Dataset Rows    | Eager Peak (KB)    | Lazy Peak (KB)     | Memory Savings 
------------------------------------------------------------------------
100             |             113.50 |             113.77 |           -0.2%
1000            |             810.62 |             196.24 |           75.8%
10000           |            7839.66 |             208.14 |           97.3%


## Conclusion & Key Takeaways

- **Eager Loading Complexity**: Grows linearly $O(N)$ with dataset size (up to ~7.8 MB for 10,000 rows).
- **Lazy Loading Complexity**: Remains constant $O(\text{Batch Size}) \approx 200 \text{ KB}$ regardless of total rows.
- **Memory Savings**: Over **97% memory savings** on 10,000 row datasets.